# Walkthrough: prior_MAP.ipynb

**Statistisches Lernen 2 — FH Kufstein Tirol**  
Arquivo de referência: `prior_MAP.ipynb`

Este walkthrough explica **passo a passo** cada célula do notebook original: o que foi feito, por que cada decisão foi tomada, como cada ferramenta funciona, e como proceder ao usá-las em situações similares.

---

> **Estrutura de cada seção:** O que o código faz → Como a ferramenta funciona → Por que foi feito assim → Como proceder / adaptar

---

## Visão geral do notebook original

O `prior_MAP.ipynb` demonstra **geometricamente** como um prior Gaussiano muda a estimativa de parâmetros. Em vez de apenas calcular números, ele visualiza a **superfície de probabilidade** no espaço dos parâmetros $(\theta_0, \theta_1)$, mostrando onde o Likelihood, o Prior e o Posterior concentram sua massa.

| Etapa | O que acontece |
|-------|----------------|
| 1 | Gerar dados com outlier deliberado |
| 2 | Montar um grid no espaço dos parâmetros |
| 3 | Calcular Likelihood, Prior e Posterior em todo o grid |
| 4 | Calcular MLE e MAP via fórmulas fechadas |
| 5 | Visualizar as três superfícies com contour plots |
| 6 | Comparar previsões e MSE no teste |

**Ideia central:** o notebook trabalha no **espaço dos parâmetros** $(\theta_0, \theta_1)$, não no espaço dos dados $(x, y)$. Cada ponto do grid representa um modelo diferente; a cor representa quão provável aquele modelo é dado os dados (Likelihood) ou o prior.

---

## Seção 1 — Intuição: por que um prior zero ajuda mesmo quando está errado?

O notebook explica algo contra-intuitivo:

> *O prior está centrado em $(0, 0)$. Os parâmetros verdadeiros são $(0.4, 1.0)$. Mesmo assim, o prior melhora a estimativa.*

### Por que isso acontece?

Com **poucos dados e um outlier**, o MLE (só dados) é puxado para longe dos parâmetros verdadeiros — o outlier tem peso desproporcional. O prior zero-centrado age como uma âncora: mesmo que esteja "errado", ele resiste ao overfitting, trazendo os parâmetros de volta para valores menores em magnitude.

```
Parâmetros verdadeiros:   (0.4, 1.0)
MLE (só dados):           (0.616, 1.335)   ← puxado pelo outlier
MAP (prior zero-centrado): (0.504, 1.111)  ← mais próximo do verdadeiro
Prior mean:               (0.0, 0.0)       ← está "errado", mas ainda ajuda
```

**Analogia:** imagina que você não sabe o peso exato de algo, mas sabe que objetos normais pesam entre 0 e 5 kg. Mesmo que esse prior seja impreciso, ele vai evitar que você estime 50 kg por causa de uma balança com defeito (outlier).

### Três conceitos-chave que o notebook demonstra

| Conceito | Onde está no espaço de parâmetros |
|----------|----------------------------------|
| **MLE** | Pico do Likelihood |
| **Prior mean** | Pico do Prior (em (0,0)) |
| **MAP** | Pico do Posterior = Likelihood × Prior |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=3, suppress=True)

# Demonstração visual da intuição ANTES de ver o código completo
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

x_demo = np.linspace(-2, 2, 200)
theta_true = np.array([0.4, 1.0])
x_tr = np.array([-1.5, -0.9, -0.3, 0.2, 0.8, 1.6])
y_tr = theta_true[0] + theta_true[1]*x_tr + np.array([0.03, -0.25, 0.24, -0.03, 0.02, 1.25])

theta_mle_demo = np.array([0.616, 1.335])
theta_map_demo = np.array([0.504, 1.111])

for ax, (theta, label, cor) in zip(
        [axes[0], axes[1], axes[2]],
        [(theta_mle_demo, 'MLE (só dados)', 'tab:red'),
         (theta_map_demo, 'MAP (com prior zero)', 'tab:orange'),
         (theta_true,     'Verdadeiro', 'black')]):
    ax.scatter(x_tr, y_tr, s=60, color='tab:blue', zorder=3)
    ax.scatter(x_tr[-1], y_tr[-1], s=120, color='red', marker='*', zorder=4, label='outlier')
    ax.plot(x_demo, theta[0] + theta[1]*x_demo, color=cor, lw=2.5, label=label)
    ax.plot(x_demo, theta_true[0] + theta_true[1]*x_demo, 'k--', lw=1, alpha=0.4, label='verdadeiro')
    ax.set_title(f'{label}\nθ=({theta[0]:.3f}, {theta[1]:.3f})', fontsize=10)
    ax.set_ylim(-3, 4); ax.legend(fontsize=8); ax.grid(alpha=0.3)

plt.suptitle('Intuição: MLE vs MAP — o prior zero reduz o efeito do outlier', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

---

## Seção 2 — Gerar os Dados

### O que o código faz

```python
true_theta = np.array([0.4, 1.0])

x_train = np.array([-1.5, -0.9, -0.3, 0.2, 0.8, 1.6])
y_clean_train = true_theta[0] + true_theta[1] * x_train
noise = np.array([0.03, -0.25, 0.24, -0.03, 0.02, 1.25])   # último é outlier
y_train = y_clean_train + noise

x_test = np.linspace(-1.8, 1.8, 200)
y_test_clean = true_theta[0] + true_theta[1] * x_test
```

---

### Decisões de design: por que estes dados específicos?

**1. `true_theta = (0.4, 1.0)` — propositalmente diferente de `(0, 0)`**  
Se os parâmetros verdadeiros fossem zero, o prior zero-centrado seria trivialmente correto. Ao usar $(0.4, 1.0)$, o notebook mostra que o prior ainda ajuda mesmo quando está errado — o ponto é não sobre acertar os parâmetros exatos, mas sobre resistir ao overfitting.

**2. Outlier no último ponto: `noise[-1] = 1.25` com `x[-1] = 1.6` (high-leverage)**  
O ponto mais distante da média de $x$ tem o maior **leverage** — influência desproporcional na estimativa da inclinação. Colocar o outlier em $x = 1.6$ (o mais extremo) maximiza seu efeito na inclinação $\theta_1$, tornando o contraste MLE vs MAP mais visível.

**Leverage** é uma propriedade dos pontos de treino independente de $y$:
$$h_i = x_i^T (X^TX)^{-1} x_i$$
Pontos com $h_i$ alto distorcem mais a regressão quando têm resíduos grandes.

**3. Teste limpo: `x_test` sem outliers**  
O conjunto de teste segue a tendência verdadeira sem ruído. Isso permite medir se o modelo generaliza para a realidade, não para o outlier de treino. É uma separação propositalmente justa.

In [ ]:
# Reproduzindo a geração de dados do notebook original
true_theta = np.array([0.4, 1.0])

x_train = np.array([-1.5, -0.9, -0.3, 0.2, 0.8, 1.6])
y_clean_train = true_theta[0] + true_theta[1] * x_train
noise = np.array([0.03, -0.25, 0.24, -0.03, 0.02, 1.25])
y_train = y_clean_train + noise

x_test = np.linspace(-1.8, 1.8, 200)
y_test_clean = true_theta[0] + true_theta[1] * x_test

print("=== Dados de treino ===")
print(f"  x_train:       {x_train}")
print(f"  y_clean_train: {y_clean_train.round(3)}  (tendência verdadeira)")
print(f"  noise:         {noise}")
print(f"  y_train:       {y_train.round(3)}  (com ruído)")
print()
print("Análise do leverage (influência de cada ponto na regressão):")
X_tr = np.column_stack([np.ones_like(x_train), x_train])
XTX_inv = np.linalg.inv(X_tr.T @ X_tr)
leverages = np.array([X_tr[i] @ XTX_inv @ X_tr[i] for i in range(len(x_train))])
for i, (xi, hi, ni) in enumerate(zip(x_train, leverages, noise)):
    marker = " ← OUTLIER (alto leverage + alto ruído)" if i == 5 else ""
    print(f"  x={xi:+.1f}  leverage={hi:.3f}  ruído={ni:+.2f}{marker}")

plt.figure(figsize=(8, 5))
plt.scatter(x_train, y_train, s=80, color='tab:blue', zorder=3, label='dados de treino')
plt.scatter(x_train[-1], y_train[-1], s=150, color='red', marker='*', zorder=4, label='outlier')
plt.plot(x_test, y_test_clean, 'k--', lw=2, label=f'tendência verdadeira θ=({true_theta[0]}, {true_theta[1]})')
plt.xlabel('x'); plt.ylabel('y')
plt.title('Dataset: poucos pontos + outlier de alto leverage')
plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

---

## Seção 3 — O Espaço dos Parâmetros: Likelihood, Prior e Posterior no Grid

### O que o código faz — a ideia central

```python
theta0_grid = np.linspace(-0.6, 1.6, 300)
theta1_grid = np.linspace(-0.1, 2.3, 300)
T0, T1 = np.meshgrid(theta0_grid, theta1_grid)
```

Em vez de calcular o Likelihood apenas nos pontos MLE e MAP, o notebook avalia a função em **90.000 combinações** de $(\theta_0, \theta_1)$ — criando uma imagem da paisagem probabilística completa.

---

### Como funciona: `np.meshgrid`

`np.meshgrid(a, b)` transforma dois vetores 1D em duas matrizes 2D que formam um grid:

```python
a = [1, 2, 3]   # valores para eixo x
b = [10, 20]    # valores para eixo y

T0, T1 = np.meshgrid(a, b)

# T0 (repete as linhas de a):
# [[1, 2, 3],
#  [1, 2, 3]]

# T1 (repete as colunas de b):
# [[10, 10, 10],
#  [20, 20, 20]]
```

Cada posição `(i, j)` no grid representa o ponto `(T0[i,j], T1[i,j])` no espaço de parâmetros. Isso permite avaliar qualquer função de $(\theta_0, \theta_1)$ para todas as combinações de uma vez, sem loops.

---

### Como funciona: `sse_on_grid` — broadcasting 3D

```python
def sse_on_grid(T0, T1, x, y):
    residuals = y[:, None, None] - (T0[None,:,:] + T1[None,:,:] * x[:, None, None])
    return np.sum(residuals**2, axis=0)
```

Este é o uso mais sofisticado de **NumPy broadcasting** do notebook. Veja as dimensões:

| Tensor | Shape antes | Shape depois do `None` | Significado |
|--------|------------|----------------------|-------------|
| `y` | `(6,)` | `(6, 1, 1)` | 6 amostras, 1 linha grid, 1 coluna grid |
| `x` | `(6,)` | `(6, 1, 1)` | igual |
| `T0` | `(300, 300)` | `(1, 300, 300)` | 1 amostra, 300×300 grid |
| `T1` | `(300, 300)` | `(1, 300, 300)` | igual |

Após o broadcast, `residuals` tem shape `(6, 300, 300)`: para cada uma das 6 amostras, calculamos o resíduo em cada um dos 90.000 pontos do grid. Somar no `axis=0` colapsa as amostras, dando o SSE total em cada ponto: shape `(300, 300)`.

**Por que `None`?** `None` em um índice de array insere uma dimensão de tamanho 1, equivalente a `np.newaxis`. É necessário para que o broadcasting alinhe as dimensões corretamente.

---

### Da SSE para Likelihood: a suposição Gaussiana

Assumimos que o ruído $\varepsilon \sim \mathcal{N}(0, \sigma^2)$. Então:

$$p(y_i \mid x_i, \theta) = \frac{1}{\sqrt{2\pi\sigma^2}} \exp\!\left(-\frac{(y_i - \theta^T x_i)^2}{2\sigma^2}\right)$$

Para N amostras independentes, a likelihood total é o produto:

$$p(\mathbf{y} \mid X, \theta) = \prod_{i=1}^N p(y_i \mid x_i, \theta) \propto \exp\!\left(-\frac{\text{SSE}(\theta)}{2\sigma^2}\right)$$

```python
neg_log_likelihood = SSE / (2 * sigma_noise**2)
```

---

### O Prior Gaussiano zero-centrado

$$p(\theta) = \mathcal{N}(\theta \mid 0, \sigma_{\text{prior}}^2 I) \propto \exp\!\left(-\frac{\lambda}{2}\|\theta\|^2\right)$$

```python
neg_log_prior = lambda_l2 * (T0**2 + T1**2) / 2
```

Geometricamente, este prior é uma **paraboloide** centrada em $(0, 0)$: mínimo (máxima probabilidade) em $(0,0)$, crescendo igualmente em todas as direções. As curvas de nível formam **círculos concêntricos**.

---

### O Posterior = Likelihood × Prior

Pelo Teorema de Bayes: $p(\theta \mid \mathbf{y}) \propto p(\mathbf{y} \mid \theta) \cdot p(\theta)$

No espaço log (onde produtos viram somas):
$$-\log p(\theta \mid \mathbf{y}) = \underbrace{\frac{\text{SSE}}{2\sigma^2}}_{\text{neg-log-likelihood}} + \underbrace{\frac{\lambda}{2}\|\theta\|^2}_{\text{neg-log-prior}}$$

```python
neg_log_posterior = neg_log_likelihood + neg_log_prior
```

---

### Truque de estabilidade numérica: subtrair o mínimo

```python
likelihood = np.exp(-(neg_log_likelihood - np.min(neg_log_likelihood)))
```

Sem subtrair o mínimo, `neg_log_likelihood` pode ter valores muito grandes, fazendo `np.exp(-valor)` resultar em `0.0` (underflow numérico). Subtrair o mínimo garante que:
- O máximo da likelihood sempre seja `exp(0) = 1.0`
- Os outros valores sejam relativos ao máximo
- Todos os valores fiquem no intervalo $[0, 1]$

Matematicamente, isso é multiplicar por uma constante positiva — não muda onde está o pico (MLE/MAP), apenas a escala.

In [ ]:
# Reproduzindo a construção do grid e das superfícies

sigma_noise = 0.8
lambda_l2   = 2.0

theta0_grid = np.linspace(-0.6, 1.6, 300)
theta1_grid = np.linspace(-0.1, 2.3, 300)
T0, T1 = np.meshgrid(theta0_grid, theta1_grid)

print("=== Grid no espaço dos parâmetros ===")
print(f"  theta0_grid: {len(theta0_grid)} valores de {theta0_grid[0]:.1f} a {theta0_grid[-1]:.1f}")
print(f"  theta1_grid: {len(theta1_grid)} valores de {theta1_grid[0]:.1f} a {theta1_grid[-1]:.1f}")
print(f"  T0.shape: {T0.shape}  ← 300×300 = {T0.size:,} pontos no grid")
print(f"  T1.shape: {T1.shape}")

print()
print("--- Demonstração de meshgrid com grid pequeno ---")
a = np.array([1.0, 2.0, 3.0])
b = np.array([10.0, 20.0])
A, B = np.meshgrid(a, b)
print(f"a = {a}")
print(f"b = {b}")
print(f"A (T0 equivalente):\n{A}")
print(f"B (T1 equivalente):\n{B}")
print(f"Cada par (A[i,j], B[i,j]) é um ponto no grid:")
for i in range(2):
    for j in range(3):
        print(f"  ({A[i,j]}, {B[i,j]})", end="  ")
    print()

print()
print("--- Demonstração de broadcasting 3D em sse_on_grid ---")
def sse_on_grid(T0, T1, x, y):
    # Broadcasting: (6,1,1) - (1,300,300) + (1,300,300)*(6,1,1) → (6,300,300)
    residuals = y[:, None, None] - (T0[None,:,:] + T1[None,:,:] * x[:, None, None])
    return np.sum(residuals**2, axis=0)   # somar sobre as 6 amostras

SSE = sse_on_grid(T0, T1, x_train, y_train)
print(f"SSE.shape: {SSE.shape}  ← um valor de SSE para cada ponto do grid")
print(f"SSE.min(): {SSE.min():.4f}  ← menor SSE (perto do MLE)")
print(f"SSE.max(): {SSE.max():.4f}  ← maior SSE (longe do MLE)")

neg_log_likelihood = SSE / (2 * sigma_noise**2)
neg_log_prior      = lambda_l2 * (T0**2 + T1**2) / 2
neg_log_posterior  = neg_log_likelihood + neg_log_prior

print()
print("--- Truque numérico: subtrair o mínimo ---")
print(f"neg_log_likelihood.min(): {neg_log_likelihood.min():.2f}")
print(f"neg_log_likelihood.max(): {neg_log_likelihood.max():.2f}")
print(f"np.exp(-max) sem subtração: {np.exp(-neg_log_likelihood.max()):.2e}  ← underflow!")
shifted = neg_log_likelihood - neg_log_likelihood.min()
print(f"np.exp(-max) com subtração: {np.exp(-shifted.max()):.4f}  ← Ok")

likelihood  = np.exp(-(neg_log_likelihood - neg_log_likelihood.min()))
prior       = np.exp(-(neg_log_prior      - neg_log_prior.min()))
posterior   = np.exp(-(neg_log_posterior  - neg_log_posterior.min()))

print()
print("Superfícies criadas:")
print(f"  likelihood.shape: {likelihood.shape}  valores em [{likelihood.min():.3f}, {likelihood.max():.3f}]")
print(f"  prior.shape:      {prior.shape}  valores em [{prior.min():.3f}, {prior.max():.3f}]")
print(f"  posterior.shape:  {posterior.shape}  valores em [{posterior.min():.3f}, {posterior.max():.3f}]")

---

## Seção 4 — MLE e MAP: Fórmulas Fechadas

### O que o código faz

```python
X_train = np.column_stack([np.ones_like(x_train), x_train])

# MLE: Ordinary Least Squares
mle_theta = np.linalg.solve(X_train.T @ X_train, X_train.T @ y_train)

# MAP: Ridge Regression
map_theta = np.linalg.solve(
    X_train.T @ X_train + lambda_l2 * sigma_noise**2 * np.eye(2),
    X_train.T @ y_train
)
```

---

### Derivação da fórmula MAP

O MAP minimiza $-\log p(\theta \mid \mathbf{y})$:

$$\hat{\theta}_{\text{MAP}} = \arg\min_\theta \left[ \frac{\text{SSE}(\theta)}{2\sigma^2} + \frac{\lambda}{2}\|\theta\|^2 \right]$$

Derivando e igualando a zero:

$$\frac{\partial}{\partial \theta} \left[ \frac{1}{2\sigma^2}(\mathbf{y} - X\theta)^T(\mathbf{y} - X\theta) + \frac{\lambda}{2}\theta^T\theta \right] = 0$$

$$-\frac{1}{\sigma^2} X^T(\mathbf{y} - X\theta) + \lambda\theta = 0$$

$$X^TX\theta + \lambda\sigma^2 \theta = X^T\mathbf{y}$$

$$\boxed{\hat{\theta}_{\text{MAP}} = (X^TX + \lambda\sigma^2 I)^{-1} X^T\mathbf{y}}$$

**A diferença em relação ao MLE** é apenas a adição de $\lambda\sigma^2 I$ à matriz $X^TX$:
- MLE: `np.linalg.solve(XTX, XTy)`
- MAP: `np.linalg.solve(XTX + lambda_l2 * sigma_noise**2 * np.eye(2), XTy)`

**Interpretação geométrica de $\lambda\sigma^2 I$:**  
A adição de $\lambda\sigma^2$ à diagonal torna a matriz $X^TX + \lambda\sigma^2 I$ sempre **positiva definida** (invertível), mesmo quando $X^TX$ é singular. Esta é a principal vantagem prática do Ridge: nunca falha, mesmo com features colineares.

---

### Conexão com Ridge Regression do scikit-learn

O MAP com prior Gaussiano é **matematicamente idêntico** ao Ridge Regression:

```python
from sklearn.linear_model import Ridge
ridge = Ridge(alpha=lambda_l2 * sigma_noise**2)
ridge.fit(x_train.reshape(-1, 1), y_train)
# ridge.intercept_ ≈ map_theta[0]
# ridge.coef_[0]   ≈ map_theta[1]
```

O `alpha` do scikit-learn equivale a $\lambda\sigma^2$ do notebook.

In [ ]:
# Reproduzindo o cálculo de MLE e MAP

X_train = np.column_stack([np.ones_like(x_train), x_train])

mle_theta = np.linalg.solve(X_train.T @ X_train, X_train.T @ y_train)

map_theta = np.linalg.solve(
    X_train.T @ X_train + lambda_l2 * sigma_noise**2 * np.eye(2),
    X_train.T @ y_train
)

print("=== MLE vs MAP ===")
print(f"  Prior mean (0, 0):  [0.000, 0.000]")
print(f"  Verdadeiro:         {true_theta}")
print(f"  MLE:                {mle_theta}")
print(f"  MAP:                {map_theta}")
print()
print(f"  Distância MLE ao verdadeiro: {np.linalg.norm(mle_theta - true_theta):.5f}")
print(f"  Distância MAP ao verdadeiro: {np.linalg.norm(map_theta - true_theta):.5f}")
print(f"  MAP está {(1 - np.linalg.norm(map_theta-true_theta)/np.linalg.norm(mle_theta-true_theta))*100:.1f}% mais perto do verdadeiro")

print()
print("--- O que lambda_l2 * sigma_noise**2 faz na matriz X^T X ---")
XTX = X_train.T @ X_train
reg_term = lambda_l2 * sigma_noise**2 * np.eye(2)
print(f"  X^T X:\n{XTX}")
print(f"  λσ²·I  (lambda={lambda_l2}, sigma={sigma_noise}):\n{reg_term}")
print(f"  X^T X + λσ²·I:\n{XTX + reg_term}")
print(f"  det(X^T X):         {np.linalg.det(XTX):.4f}")
print(f"  det(X^T X + λσ²·I): {np.linalg.det(XTX + reg_term):.4f}  ← sempre positivo")

print()
print("--- Verificação: MAP com sklearn.Ridge ---")
from sklearn.linear_model import Ridge
alpha_sklearn = lambda_l2 * sigma_noise**2   # equivalência
ridge = Ridge(alpha=alpha_sklearn, fit_intercept=True)
ridge.fit(x_train.reshape(-1, 1), y_train)
print(f"  alpha sklearn = λ × σ² = {lambda_l2} × {sigma_noise}² = {alpha_sklearn}")
print(f"  MAP   (manual):  θ₀={map_theta[0]:.5f}  θ₁={map_theta[1]:.5f}")
print(f"  Ridge (sklearn): θ₀={ridge.intercept_:.5f}  θ₁={ridge.coef_[0]:.5f}")
print(f"  São iguais? {np.allclose(map_theta, [ridge.intercept_, ridge.coef_[0]], atol=1e-4)}")

print()
print("--- Efeito de lambda_l2 nos parâmetros ---")
print(f"{'lambda':>10s} | {'theta0':>8s} | {'theta1':>8s} | {'dist_verdadeiro':>15s}")
print("-" * 50)
for lam in [0.0, 0.5, 2.0, 5.0, 20.0, 100.0]:
    if lam == 0:
        th = mle_theta
        label = "MLE"
    else:
        th = np.linalg.solve(XTX + lam * sigma_noise**2 * np.eye(2), X_train.T @ y_train)
        label = f"MAP λ={lam}"
    dist = np.linalg.norm(th - true_theta)
    print(f"{label:>10s} | {th[0]:8.4f} | {th[1]:8.4f} | {dist:15.5f}")

---

## Seção 5 — Visualização: Contour Plots no Espaço dos Parâmetros

### O que o código faz

```python
def plot_heatmap(Z, title, ax, cmap='viridis'):
    im = ax.contourf(T0, T1, Z, levels=60, cmap=cmap)
    ax.contour(T0, T1, Z, levels=8, colors='white', alpha=0.35, linewidths=0.8)
    ax.scatter(*true_theta, marker='*', s=220, ...)
    ax.scatter(0, 0,        marker='o', s=90,  ...)
    ax.scatter(*mle_theta,  marker='X', s=130, ...)
    ax.scatter(*map_theta,  marker='D', s=90,  ...)
```

---

### `contourf` vs `contour` — preenchido vs linhas

| Função | O que plota | Quando usar |
|--------|-------------|-------------|
| `ax.contourf(X, Y, Z, levels=N, cmap=...)` | Regiões **preenchidas** com cor | Mostrar superfícies de probabilidade/densidade |
| `ax.contour(X, Y, Z, levels=N, colors=...)` | Apenas as **linhas** de contorno | Adicionar marcadores de curvas de nível sobre o preenchido |

**`levels`:** quantos contornos desenhar. Com `levels=60`, a transição de cor é suave. Com `levels=8`, apenas 8 linhas visíveis.

**Ordem:** sempre chame `contourf` **antes** de `contour` — as linhas brancas devem ficar por cima do preenchimento.

**`fig.colorbar(im, ax=ax, label=...)`:** a colorbar deve receber o objeto retornado por `contourf` (o `im`), não qualquer outro.

---

### Markers do `scatter` — o que cada forma significa

| `marker=` | Forma | Representa no notebook |
|-----------|-------|------------------------|
| `'*'` | Estrela grande | Parâmetros verdadeiros $\theta_{\text{true}}$ |
| `'o'` | Círculo | Prior mean $(0, 0)$ |
| `'X'` | X grande (filled) | Estimativa MLE |
| `'D'` | Diamante | Estimativa MAP |

Usar formas **diferentes** em vez de apenas cores diferentes é boa prática: garante que o gráfico seja legível mesmo em preto e branco ou para pessoas com daltonismo.

---

### `*true_theta` — desempacotar array como argumentos

```python
ax.scatter(*true_theta, ...)   # equivalente a ax.scatter(true_theta[0], true_theta[1], ...)
```

O operador `*` antes de uma lista/array **desempacota** seus elementos como argumentos posicionais separados. Útil quando você tem coordenadas em um array e precisa passá-las como `x, y` individuais.

---

### `constrained_layout=True` vs `tight_layout()`

```python
fig, axes = plt.subplots(1, 3, figsize=(18, 5), constrained_layout=True)
```

| Parâmetro | Comportamento |
|-----------|---------------|
| `constrained_layout=True` | Ajuste automático de espaçamento, inclusindo colorbars externas — mais robusto |
| `plt.tight_layout()` | Ajuste manual pós-plot — não lida tão bem com legendas externas e colorbars |

Use `constrained_layout=True` quando tiver colorbars ou legendas fora dos axes — evita sobreposições.

---

### Leitura dos três gráficos

**Likelihood only:**  
O pico (brighter) está no MLE $(0.616, 1.335)$ — deslocado pelo outlier. A região de alta probabilidade é uma elipse inclinada.

**Prior only:**  
Curvas de nível são **círculos concêntricos** centrados em $(0, 0)$. O pico está em $(0, 0)$, independente dos dados.

**Posterior:**  
É o produto dos dois. O pico está no MAP $(0.504, 1.111)$ — entre o MLE e a prior mean, mais próximo do parâmetro verdadeiro $(0.4, 1.0)$. O prior "puxou" o pico da likelihood em direção à origem.

In [ ]:
# Reproduzindo a função de visualização e todos os plots

def plot_heatmap(Z, title, ax, cmap='viridis'):
    im = ax.contourf(T0, T1, Z, levels=60, cmap=cmap)
    ax.contour(T0, T1, Z, levels=8, colors='white', alpha=0.35, linewidths=0.8)
    ax.scatter(*true_theta, marker='*', s=220, color='white',      edgecolor='black', zorder=5, label='true θ')
    ax.scatter(0, 0,        marker='o', s=90,  color='tab:green',  edgecolor='black', zorder=5, label='prior mean (0,0)')
    ax.scatter(*mle_theta,  marker='X', s=130, color='tab:red',    edgecolor='black', zorder=5, label='MLE')
    ax.scatter(*map_theta,  marker='D', s=90,  color='tab:orange', edgecolor='black', zorder=5, label='MAP')
    ax.set_xlabel('θ₀ (intercepto)')
    ax.set_ylabel('θ₁ (inclinação)')
    ax.set_title(title, fontsize=11)
    ax.grid(alpha=0.15)
    return im

# Gráficos individuais
for Z, title, cmap, cbar_label in [
    (likelihood, 'Data Likelihood apenas', 'viridis', 'likelihood relativo'),
    (prior,      'Prior Gaussiano zero-centrado', 'magma', 'densidade prior relativa'),
    (posterior,  'Posterior = Likelihood × Prior', 'cividis', 'posterior relativo'),
]:
    fig, ax = plt.subplots(figsize=(7, 6))
    im = plot_heatmap(Z, title, ax, cmap=cmap)
    fig.colorbar(im, ax=ax, label=cbar_label)
    ax.legend(loc='upper left', bbox_to_anchor=(1.35, 1.0), fontsize=9)
    plt.tight_layout()
    plt.show()

# Comparação lado a lado (igual ao notebook original)
fig, axes = plt.subplots(1, 3, figsize=(18, 5), constrained_layout=True)
plots = [
    (likelihood, 'Likelihood', 'viridis', 'likelihood relativo'),
    (prior,      'Prior',      'magma',   'prior relativo'),
    (posterior,  'Posterior',  'cividis', 'posterior relativo'),
]
for ax, (Z, title, cmap, cbar_label) in zip(axes, plots):
    im = plot_heatmap(Z, title, ax, cmap=cmap)
    fig.colorbar(im, ax=ax, label=cbar_label)

handles, labels = axes[-1].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', ncol=4, bbox_to_anchor=(0.5, -0.1), fontsize=10)
plt.suptitle('Espaço dos parâmetros: Likelihood × Prior = Posterior', fontsize=13, fontweight='bold')
plt.show()

print("\nAnálise dos picos das superfícies:")
peak_lik = (T0.flat[likelihood.argmax()], T1.flat[likelihood.argmax()])
peak_post = (T0.flat[posterior.argmax()], T1.flat[posterior.argmax()])
print(f"  Pico do Likelihood (≈ MLE):  θ=({peak_lik[0]:.3f}, {peak_lik[1]:.3f})")
print(f"  MLE calculado exato:         θ=({mle_theta[0]:.3f}, {mle_theta[1]:.3f})")
print(f"  Pico do Posterior (≈ MAP):   θ=({peak_post[0]:.3f}, {peak_post[1]:.3f})")
print(f"  MAP calculado exato:         θ=({map_theta[0]:.3f}, {map_theta[1]:.3f})")
print("  (Pequenas diferenças por resolução do grid — a fórmula analítica é exata)")

---

## Seção 6 — Comparar Previsões e MSE no Teste

### O que o código faz

```python
def predict(theta, x):
    return theta[0] + theta[1] * x

mle_mse = np.mean((mle_pred_test - y_test_clean)**2)
map_mse = np.mean((map_pred_test - y_test_clean)**2)
```

---

### Por que o teste é contra `y_test_clean` (sem ruído)?

O objetivo do modelo não é reproduzir o ruído de treino — é aprender a **tendência subjacente** $f(x) = \theta_0 + \theta_1 x$. Avaliar contra `y_test_clean` (a tendência verdadeira sem ruído) mede exatamente o que importa: quão bem o modelo aprendeu o sinal real.

Se o teste fosse contra dados ruidosos, o MSE seria sempre maior e a comparação seria menos clara.

---

### MSE como medida de generalização

$$\text{Test MSE}(\hat{\theta}) = \frac{1}{N_{\text{test}}} \sum_{i=1}^{N_{\text{test}}} (\hat{\theta}^T x_i^{\text{test}} - y_i^{\text{clean}})^2$$

O notebook mostra:
- **MLE test MSE:** maior, porque overfittou ao outlier
- **MAP test MSE:** menor, porque o prior resistiu ao outlier

**Melhoria relativa:** $\frac{\text{MSE}_{\text{MLE}} - \text{MSE}_{\text{MAP}}}{\text{MSE}_{\text{MLE}}} \times 100\%$

Esta melhoria **não é garantida sempre** — depende de:
- O outlier ser suficientemente extremo
- O `lambda_l2` estar na faixa certa
- O prior ser razoavelmente centrado

In [ ]:
# Reproduzindo a comparação final do notebook

def predict(theta, x):
    return theta[0] + theta[1] * x

mle_pred_test = predict(mle_theta, x_test)
map_pred_test = predict(map_theta, x_test)

mle_mse = np.mean((mle_pred_test - y_test_clean)**2)
map_mse = np.mean((map_pred_test - y_test_clean)**2)

print("=== Test MSE ===")
print(f"  MLE test MSE: {mle_mse:.4f}")
print(f"  MAP test MSE: {map_mse:.4f}")
print(f"  Melhoria relativa: {(mle_mse - map_mse)/mle_mse*100:.1f}%")

plt.figure(figsize=(9, 5))
plt.scatter(x_train, y_train, s=80, color='tab:blue', zorder=3, label='treino')
plt.scatter(x_train[-1], y_train[-1], s=150, color='red', marker='*', zorder=4, label='outlier')
plt.plot(x_test, y_test_clean,   'k--', lw=2, label='tendência verdadeira')
plt.plot(x_test, mle_pred_test, color='tab:red',    lw=2,
         label=f'MLE  (test MSE={mle_mse:.4f})')
plt.plot(x_test, map_pred_test, color='tab:orange', lw=2,
         label=f'MAP  (test MSE={map_mse:.4f})')
plt.xlabel('x'); plt.ylabel('y')
plt.title('Prior zero-centrado melhora a generalização reduzindo o overfitting')
plt.legend(fontsize=9); plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# Varredura de lambda_l2 para ver efeito no test MSE
lambdas = np.logspace(-3, 2, 50)
mse_por_lambda = []
for lam in lambdas:
    th = np.linalg.solve(X_train.T @ X_train + lam * sigma_noise**2 * np.eye(2),
                          X_train.T @ y_train)
    mse_por_lambda.append(np.mean((predict(th, x_test) - y_test_clean)**2))

melhor_lambda = lambdas[np.argmin(mse_por_lambda)]

fig, ax = plt.subplots(figsize=(9, 4))
ax.semilogx(lambdas, mse_por_lambda, 'b-', lw=2, label='Test MSE (MAP)')
ax.axhline(mle_mse, color='red', ls='--', lw=1.5, label=f'MLE test MSE={mle_mse:.4f}')
ax.axvline(lambda_l2, color='orange', ls=':', lw=2, label=f'λ do notebook ({lambda_l2})')
ax.axvline(melhor_lambda, color='green', ls='--', lw=1.5, label=f'λ* ótimo={melhor_lambda:.3f}')
ax.set_xlabel('λ (escala log)'); ax.set_ylabel('Test MSE')
ax.set_title('Test MSE em função de λ — existe um λ ótimo')
ax.legend(fontsize=9); ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nλ usado no notebook: {lambda_l2}  →  test MSE = {mse_por_lambda[np.searchsorted(lambdas, lambda_l2)]:.4f}")
print(f"λ* ótimo:            {melhor_lambda:.3f}  →  test MSE = {min(mse_por_lambda):.4f}")
print(f"MLE (λ=0):                   →  test MSE = {mle_mse:.4f}")

---

## Seção 7 — "Play around": o efeito de `lambda_l2`

O notebook termina com a instrução: *"try changing `lambda_l2`"*. Aqui vemos exatamente o que acontece:

| `lambda_l2` | Comportamento |
|-------------|---------------|
| `0.0` | MAP = MLE (sem prior, só dados) |
| `0.5–2.0` | Shrinkage benéfico, MAP próximo do verdadeiro |
| `5.0–10.0` | MAP começa a se aproximar do prior mean (0,0) |
| `100+` | MAP ≈ (0,0), underfitting severo |

Este é exatamente o **Bias-Variance Trade-Off** da regularização:
- $\lambda$ pequeno → baixo bias, alta variância (sensível ao outlier)
- $\lambda$ grande → alto bias (puxado para zero), baixa variância
- $\lambda^*$ ótimo → equilíbrio, minimiza o test MSE

In [ ]:
# Demonstração: efeito de diferentes lambda_l2 nas retas ajustadas

lambdas_demo = [0.0, 0.5, 2.0, 10.0, 100.0]
cores_demo   = ['tab:red', 'tab:purple', 'tab:orange', 'tab:green', 'tab:gray']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.scatter(x_train, y_train, s=70, color='tab:blue', zorder=4)
ax.scatter(x_train[-1], y_train[-1], s=130, color='red', marker='*', zorder=5)
ax.plot(x_test, y_test_clean, 'k--', lw=2, alpha=0.5, label='verdadeiro')

for lam, cor in zip(lambdas_demo, cores_demo):
    if lam == 0:
        th = mle_theta
        lbl = f'λ=0 (MLE)'
    else:
        th = np.linalg.solve(X_train.T @ X_train + lam * sigma_noise**2 * np.eye(2),
                              X_train.T @ y_train)
        lbl = f'λ={lam}'
    ax.plot(x_test, predict(th, x_test), color=cor, lw=2, label=lbl)

ax.set_title('Retas ajustadas por λ')
ax.set_xlabel('x'); ax.set_ylabel('y')
ax.legend(fontsize=9); ax.grid(alpha=0.3)

# No espaço dos parâmetros
ax = axes[1]
ax.contourf(T0, T1, likelihood, levels=30, cmap='Blues', alpha=0.5)
ax.scatter(*true_theta, marker='*', s=300, color='black',      zorder=5, label='verdadeiro')
ax.scatter(0, 0,        marker='o', s=120, color='tab:green',  zorder=5, label='prior mean')

thetas_demo = []
for lam, cor in zip(lambdas_demo, cores_demo):
    if lam == 0:
        th = mle_theta
        lbl = f'λ=0 (MLE)'
    else:
        th = np.linalg.solve(X_train.T @ X_train + lam * sigma_noise**2 * np.eye(2),
                              X_train.T @ y_train)
        lbl = f'λ={lam}'
    thetas_demo.append(th)
    ax.scatter(*th, marker='D', s=100, color=cor, edgecolor='black', zorder=5, label=lbl)

# Traçar o caminho dos MAP conforme lambda cresce
path = np.array(thetas_demo)
ax.plot(path[:, 0], path[:, 1], 'w--', lw=1.5, alpha=0.7, zorder=4)

ax.set_xlabel('θ₀ (intercepto)'); ax.set_ylabel('θ₁ (inclinação)')
ax.set_title('Caminho do MAP no espaço dos parâmetros\n(λ cresce → MAP se aproxima de (0,0))')
ax.legend(fontsize=8, loc='upper right'); ax.grid(alpha=0.2)

plt.suptitle('Efeito de λ: Bias-Variance Trade-Off na regularização L2', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print("Tabela: θ estimado por λ")
print(f"{'λ':>8s} | {'θ₀':>8s} | {'θ₁':>8s} | {'dist_verdadeiro':>15s} | Interpretação")
print('-' * 65)
interpretacoes = ['Só dados (overfitting)', 'Shrinkage suave', 'Shrinkage moderado (notebook)',
                   'Shrinkage forte', 'Underfitting (≈ prior)']
for lam, cor, interp in zip(lambdas_demo, cores_demo, interpretacoes):
    if lam == 0:
        th = mle_theta
    else:
        th = np.linalg.solve(X_train.T @ X_train + lam * sigma_noise**2 * np.eye(2),
                              X_train.T @ y_train)
    dist = np.linalg.norm(th - true_theta)
    print(f"{lam:>8.1f} | {th[0]:8.4f} | {th[1]:8.4f} | {dist:15.5f} | {interp}")

---

## Guia de ferramentas: referência rápida

### NumPy — ferramentas usadas

| Ferramenta | Sintaxe | O que faz |
|------------|---------|----------|
| Grid 2D | `np.meshgrid(a, b)` | Cria matrizes T0, T1 para avaliar funções em grade 2D |
| Broadcast 3D | `x[:, None, None]` | Insere dimensões extras para broadcasting |
| Estabilidade exp | `np.exp(-(Z - Z.min()))` | Evita underflow ao exponentiar arrays grandes |
| Solve MAP | `np.linalg.solve(XTX + λσ²I, XTy)` | Resolve as Normal Equations regularizadas |
| Matriz identidade | `np.eye(n)` | Matriz identidade n×n |
| Norma vetorial | `np.linalg.norm(v)` | ‖v‖ — distância euclidiana |
| Precisão impressão | `np.set_printoptions(precision=3, suppress=True)` | Exibe 3 casas decimais, sem notação científica |

### Matplotlib — ferramentas usadas

| Ferramenta | Sintaxe | O que faz |
|------------|---------|----------|
| Mapa de cores preenchido | `ax.contourf(X, Y, Z, levels=N, cmap=...)` | Plota superfície 2D com cores |
| Linhas de contorno | `ax.contour(X, Y, Z, levels=N, colors=...)` | Plota curvas de nível |
| Colorbar | `fig.colorbar(im, ax=ax, label=...)` | Barra de cores para contourf |
| Layout constrained | `plt.subplots(..., constrained_layout=True)` | Evita sobreposição com colorbars externas |
| Desempacotar array | `ax.scatter(*theta, ...)` | Passa elementos do array como x, y separados |
| Marker formas | `marker='*', 'o', 'X', 'D'` | Estrela, círculo, X preenchido, diamante |
| Legenda externa | `ax.legend(bbox_to_anchor=(1.2, 1.0))` | Move legenda para fora do axes |
| Legenda de figura | `fig.legend(handles, labels, ...)` | Legenda compartilhada entre múltiplos axes |

---

## Quando usar esta abordagem (espaço de parâmetros)

A visualização no **espaço dos parâmetros** é especialmente útil quando:
- O modelo tem **2 parâmetros** (pode ser visualizado em 2D)
- Você quer entender **geometricamente** como o prior interage com a likelihood
- Você está explicando regularização para alguém novo no assunto

Para modelos com mais de 2 parâmetros, a mesma lógica se aplica, mas você só pode visualizar **projeções** 2D (fixando os outros parâmetros).

---

## Exercícios para consolidar

1. **Mude o prior:** substitua o prior zero-centrado por um prior centrado nos parâmetros verdadeiros $(0.4, 1.0)$. O MAP melhora ainda mais?
2. **Remova o outlier:** defina `noise[-1] = 0.02`. O MLE e o MAP convergem? O prior ainda ajuda?
3. **Prior de Laplace:** implemente `neg_log_prior = lambda_l2 * (|T0| + |T1|)` (L1). Compare a forma das curvas de nível com L2.
4. **Mais dados:** duplique `x_train` e `y_train`. Com mais dados, o prior tem menos efeito? Por quê?
5. **Grid mais fino:** aumente de `300×300` para `500×500`. A precisão do pico grid se aproxima mais do MAP analítico?
6. **sigma_noise:** aumente `sigma_noise` de `0.8` para `2.0`. O que acontece com a forma do Likelihood? Com o MAP?